# Split and move the DPR processing flow

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-672

## 1. Initialisation

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
USE_DPR_MOCKUP = True

# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=2, use_mockup = USE_DPR_MOCKUP)
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)
display(dask_cluster_staging)

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-eopf-mockup': http://dask-eopf-mockup:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf-mockup': http://localhost:8703/clusters/fbeda5716d524e4aa643c3a47db5e080/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+---------+
| Package     | Client   | Scheduler | Workers |
+-------------+----------+-----------+---------+
| dask        | 2024.5.2 | 2025.2.0  | None    |
| distributed | 2024.5.2 | 2025.2.0  | None    |
| tornado     | 6.3.3    | 6.4.2     | None    |
+-------------+----------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf-mockup' are up: 0/2
Dask workers for 'dask-eopf-mockup' are up: 2/2
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Create new dask cluster
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/a8fe02908b3646c8aa97f520bf0b684d/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-staging' are up: 0/2
Dask workers for 'dask-staging' are up: 2/2


In [37]:
# Create a test collection
CATALOG_COLLECTION_ID = "SPRINT24_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)

# Check that it is empty
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
assert not list(items)

# Other test values
SESSION_ID = "S1A_20200105072204051312"
CADIP_COLLECTION_ID = "sgs_sentinel1"

15:13:37.352 [INFO] (rs_client.rs_client) Retrieving all items from collection 'jgaucher:SPRINT24_TEST_COLLECTION'.


In [38]:
# Other imports
from contextlib import chdir
import os
import os.path as osp
import prefect
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import rs_workflows
from rs_workflows.flow_utils import FlowEnvSerialized

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

flow_parameters = {
  "extra_args": {
    "owner_id": OWNER_ID,
  },
  "cadip_collection_identifier": CADIP_COLLECTION_ID,
  "session_identifier": SESSION_ID,
  "catalog_collection_identifier": CATALOG_COLLECTION_ID
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

15:13:37.834 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/logging_config.yaml'.

15:13:37.836 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml'.

15:13:37.838 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_3A.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_3A.yaml'.

15:13:37.839 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_dpr_mockup.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'.

15:13:37.860 | INFO    | prefect.S3Bucket - Uploaded 4 files from 'l0/config' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [70]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 
workflows_folder = f"{s3_code_folder}/rs_workflows"

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload workflows package and resources contents
with chdir(Path(rs_workflows.__file__).parent.parent):
    await share_bucket.put_directory(local_path = "rs_workflows", to_path = workflows_folder)
await share_bucket.put_directory(local_path = "../../resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/code'


In [71]:
# Deploy the flows
for entrypoint, name, deploy_name in [
    [
        "cadip_flow.py:search_and_stage",
        "Cadip search and stage",
        "cadip-search-stage/Cadip search and stage",
    ],
    # Declare the main flow at the end
    [
        "on_demand_processing.py:on_demand_processing", 
        "On-demand processing",
        "on-demand-processing/On-demand processing"
    ],
]:
    flow = await prefect.flow.from_source(
        source=share_bucket,
        entrypoint=f"{workflows_folder}/{entrypoint}",
    )
    await flow.deploy(
        name=name,
        work_pool_name=os.environ["PREFECT_WORK_POOL_EOPF"], 
        tags=["demo", "sprint 24"],
        ignore_warnings=True,
    )
    await prefect_utils.wait_for_deployment(deploy_name)

Output()

Successfully created/updated all deployments!

                           Deployments                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ Name                                      ┃ Status  ┃ Details ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩
│ cadip-search-stage/Cadip search and stage │ applied │         │
└───────────────────────────────────────────┴─────────┴─────────┘

To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'cadip-search-stage/Cadip search and stage'

...

                           Deployments                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ Name                                      ┃ Status  ┃ Details ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩
│ on-demand-processing/On-demand processing │ applied │         │
└───────────────────────────────────────────┴─────────┴─────────┘

To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'on-demand-processing/On-demand processing'

You can also run your flow via the Prefect UI: http://prefect-server:4200/deployments/deployment/1035037d-4ac3-4cab-b26e-4e8a0cbe3e29

Finished deploying prefect flow: 'on-demand-processing/On-demand processing'


## 3. Run Prefect flow

In [72]:
# Convert to json to trigger prefect flow
params_str = to_json(flow_parameters) # flow parameters

In [73]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 'on-demand-processing/On-demand processing'...
Created flow run 'venomous-herring'.
└── UUID: 47283c30-dc01-49f9-b10e-0e64e106e72e
└── Parameters: {'extra_args': {'owner_id': 'jgaucher'}, 'cadip_collection_identifier': 'sgs_sentinel1', 'session_identifier': 'S1A_20200105072204051312', 'catalog_collection_identifier': 'SPRINT24_TEST_COLLECTION'}
└── Job Variables: {}
└── Scheduled start time: 2025-05-26 15:37:19 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/47283c30-dc01-49f9-b10e-0e64e106e72e
Watching flow run 'venomous-herring'...


15:37:20.538 | INFO    | prefect - Flow run is in state 'Pending'
15:37:22.433 | INFO    | prefect - Flow run is in state 'Running'
15:37:28.002 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


In [57]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)
eopf_prod_ids = ["S03MWRL0__20221101T092439_6037_A307_T677", "S03OLCL0__20210629T044945_0119_A247_T219"]
for id in eopf_prod_ids:
    assert catalog_client.get_item(TEST_COLLECTION_NAME, id) 
   

NameError: name 'output_data_dir' is not defined

## 6. Shutdown the dask clusters

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)
    dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [7]:
from importlib import reload
debug_flow = True

In [ ]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway_staging, None)
    shutdown_dask_clusters(dask_gateway_eopf, None)
    init_dask_cluster_eopf(scale=2)
    init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *

In [48]:
if debug_flow:

    import sys
    from rs_workflows import on_demand_processing    

    # from rs_workflows.on_demand_processing import CadipFlowParams

    # Reload the flow and all rs-client-libraries modules
    reload(on_demand_processing)
    for module in list(sys.modules.values()):
        if any(module.__name__.startswith(prefix) for prefix in ["rs_client.", "rs_common.", "rs_workflows."]):
            reload(module)

    results = await on_demand_processing.on_demand_processing(**flow_parameters)
    display(results)

15:21:04.323 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/d8c91074-9f89-42a2-b0bd-6ac81f4b65a3

15:21:04.354 | INFO    | Flow run 'offbeat-grouse' - Beginning flow run 'offbeat-grouse' for flow 'on-demand-processing'

15:21:04.355 | INFO    | Flow run 'offbeat-grouse' - View at http://prefect-server:4200/runs/flow-run/d8c91074-9f89-42a2-b0bd-6ac81f4b65a3

15:21:04.397 | WARNING | opentelemetry.trace - Overriding of current TracerProvider is not allowed

15:21:04.401 | WARNING | opentelemetry.instrumentation.instrumentor - Attempting to instrument while already instrumented

15:21:04.403 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"

15:21:14.559 | INFO    | Flow run 'offbeat-grouse' - Finished in state Completed()

None